In [45]:
import pandas as pd 

#../ sind nötig, damit zuerst einmal aus dem notebook Ordner raus und dann in den data ordner rein.
fire_data = pd.read_csv("../data/raw/NFDB_point.csv", sep =";")

In [15]:
##Data cleaning
#Renaming Columns for data of Canada
new_names = {"SRC_AGENCY": "province",
            "LONGITUDE": "longitude",
            "LATITUDE": "latitude",
            "SIZE_HA": "size_ha",
            "FIRE_ID": "fire_id",
            "FID": "fid",
            "CAUSE": "cause",
            "YEAR": "year"}

#rename operation with the parameter "columns"
fire_data = fire_data.rename(columns=new_names) 

In [33]:
#errors = "ignore" ignore wird dazu verwendet dass wenn panda den code 
#nochmals laden will, dass die Spalten gar nicht mehr gefunden werden können 
#und daher kann man mit errors=ignore diese Meldung ignorieren falls es eine gibt
fire_data = fire_data.drop(
    columns=["the_geom",
             "NFDBFIREID",
             "NAT_PARK",
             "FIRENAME",
             "REP_DATE",
             "OUT_DATE",
             "FIRE_TYPE",
             "RESPONSE",
             "PROTZONE",
             "MORE_INFO",
             "MONTH",
             "DAY"],
    errors="ignore") 

#make a safe copy
fire_data = fire_data.copy()

In [44]:
#Abkürzungen für Provinzen umwandeln in Namen
#Zuteilung der Provinznamen zu den Abkürzungen
province = {
    "ON": "Ontario",
    "BC": "British Columbia",
    "QC": "Quebec",
    "AB": "Alberta",
    "MB": "Manitoba",
    "SK": "Saskatchewan",
    "NS": "Nova Scotia",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "PE": "Prince Edward Island",
    "YT": "Yukon",
    "NT": "Northwest Territories",
    "NU": "Nunavut"}

#Ersetzen der Abkürzungen mit dem vollen Namen. 
fire_data["province"] = fire_data["province"].map(province).fillna(fire_data["province"])

In [18]:
#Abkürzungen für Ursache(cause) umwandeln in ausgeschriebenen Name
#Zuteilung der Provinznamen zu den Abkürzungen
CAUSE_ENTIRE = {
    "H": "Human", 
    "H-PB": "Prescribed Burn",
    "U": "Unknown", 
    "N": "Natural"}

#Ersetzen der Abkürzungen mit dem vollen Namen. 
fire_data["cause"] = fire_data["cause"].map(CAUSE_ENTIRE).fillna(fire_data["cause"])

In [19]:
#Fokus nur auf unkontrollierte Feuer, daher alle Zeilen mit H-PB herauslöschen
#bestimmen welche Reihen gelöscht werden sollen 
drop_rows = fire_data[fire_data["cause"]=="Prescribed Burn"].index

fire_data = fire_data.drop(index = drop_rows)


In [20]:
#Daten nur für 2023 auswählen. 
fire_2023 = fire_data[fire_data["year"]==2023]

print(f"There were {len(fire_2023)} starting in the year 2023") 

There were 965 starting in the year 2023


In [21]:
#Mögliche outliers löschen
fire_2023 = fire_2023[fire_2023["longitude"] <= 0]

In [22]:
fire_2023 = fire_2023.sort_values(by="size_ha", ascending=False)

In [35]:
#Definition of a classification function

#Minimum is 200 ha and Maximum is 885388 ha.
#klein: size_ha < 500 ha, mittel: size_ha >= 500 AND < 1000
#gross: size_ha >= 1000 AND < 10000, sehr gross: size_ha >= 10000 AND < 100000 
#mega feuer: size_ha >= 100000

#Thresholds

SMALL_HA = 500
MEDIUM_HA = 1000
BIG_HA = 10000
VERY_BIG_HA = 100000

In [36]:
#Step 1
def classify_size(size_ha):
    if size_ha <= SMALL_HA:
        return "small"
    elif 500 <= size_ha < MEDIUM_HA:
        return "medium"
    elif 1000 <= size_ha < BIG_HA:
        return "big"
    elif 10000 <= size_ha < VERY_BIG_HA:
        return "very big"
    else:
        return "mega fire"

#Step 2: Neue Column erstellen 
fire_2023["size class"] = fire_2023["size_ha"].apply(classify_size)

In [37]:
#Step 3: Logical Order for the categories to prevent alphabetical sorting in the legend
CATEGORY_ORDER = ["small", "medium", "big", "very big", "mega fire"]

fire_2023["size class"] = pd.Categorical(
    fire_2023["size class"], categories=CATEGORY_ORDER, ordered=True
)

In [38]:
#Step 4: to color it later in a heat map we need to give weights to the cateogries
CLASS_WEIGHTS = {
    "small": 1,
    "medium": 2,
    "big": 4,
    "very big": 16,
    "mega fire": 256
}

fire_2023["class_weight"] = fire_2023["size class"].map(CLASS_WEIGHTS)

In [39]:
fire_2023.to_csv("../data/processed/fire_2023_clean.csv", index=False)